In [2]:
import pandas as pd
import numpy as np
# from scipy import stats
import matplotlib.pyplot as plt
# import seaborn as sns
# cau a
df = pd.read_excel("02_BODYTEMP.xlsx")
print("Kích thước ban đầu:", df.shape)
print("Các cột:", df.columns.tolist())
print("Mô tả nhanh:\n", df.describe())


cols = ["DAY 1 - 8AM","DAY 1 - 12AM","DAY 2 - 8AM","DAY 2 - 12AM","SEX","SMOKE"]
df = df.dropna(subset=cols)
print("Kích thước sau làm khi xóa:", df.shape)

Kích thước ban đầu: (107, 6)
Các cột: ['SEX', 'SMOKE', 'DAY 1 - 8AM', 'DAY 1 - 12AM', 'DAY 2 - 8AM', 'DAY 2 - 12AM']
Mô tả nhanh:
        DAY 1 - 8AM  DAY 1 - 12AM  DAY 2 - 8AM  DAY 2 - 12AM
count    38.000000     93.000000    70.000000    106.000000
mean     98.126316     98.123656    97.488571     98.200000
std       0.755407      0.646798     0.700630      0.622896
min      96.200000     96.700000    96.000000     96.500000
25%      97.525000     97.700000    97.000000     97.800000
50%      98.200000     98.200000    97.600000     98.400000
75%      98.800000     98.600000    98.000000     98.600000
max      99.400000     99.400000    99.000000     99.600000
Kích thước sau làm khi xóa: (32, 6)


In [3]:
#cau b

plt.figure(figsize=(10,6))
long_df = df.melt(value_vars=["DAY 1 - 8AM","DAY 1 - 12AM","DAY 2 - 8AM","DAY 2 - 12AM"],
                  var_name="Timepoint", value_name="Temp")
sns.boxplot(data=long_df, x="Timepoint", y="Temp")
sns.stripplot(data=long_df, x="Timepoint", y="Temp", color="black", alpha=0.4, dodge=False)
plt.title("Phân phối nhiệt độ tại 4 thời điểm đo")
plt.ylabel("Nhiệt độ (°F)")
plt.xlabel("Thời điểm")
plt.tight_layout()
plt.show()

NameError: name 'sns' is not defined

<Figure size 720x432 with 0 Axes>

In [4]:
#cau c 
x = df["DAY 1 - 8AM"].to_numpy()
n = len(x)
mean = np.mean(x)
std = np.std(x, ddof=1)
alpha = 0.05
tcrit = stats.t.ppf(1 - alpha/2, df=n-1)
ci95 = (mean - tcrit*std/np.sqrt(n), mean + tcrit*std/np.sqrt(n))
print(f"Mean DAY1_8AM = {mean:.3f}, 95% CI = [{ci95[0]:.3f}, {ci95[1]:.3f}]")

NameError: name 'stats' is not defined

In [5]:
#cau d
male_mask = df["SEX"].astype(str).str.upper().isin(["M","NAM"])
x_m = df.loc[male_mask, "DAY 2 - 8AM"].to_numpy()
n_m = len(x_m)
mean_m = np.mean(x_m)
std_m = np.std(x_m, ddof=1)
alpha = 0.10
tcrit_m = stats.t.ppf(1 - alpha/2, df=n_m-1)
ci90_m = (mean_m - tcrit_m*std_m/np.sqrt(n_m), mean_m + tcrit_m*std_m/np.sqrt(n_m))
print(f"Mean DAY2_8AM (Nam) = {mean_m:.3f}, 90% CI = [{ci90_m[0]:.3f}, {ci90_m[1]:.3f}]")

NameError: name 'stats' is not defined

In [6]:
#cau e
df["MEAN_4"] = df[["DAY 1 - 8AM","DAY 1 - 12AM","DAY 2 - 8AM","DAY 2 - 12AM"]].mean(axis=1)
smoke_mask = df["SMOKE"].astype(str).str.upper().isin(["Y","HUT THUOC","SMOKE","CO"])
smokers = df.loc[smoke_mask]
n = len(smokers)
k = (smokers["MEAN_4"] > 98).sum()
phat = k / n if n > 0 else np.nan
alpha = 0.01
z = stats.norm.ppf(1 - alpha/2)
if n > 0:
    denom = 1 + z**2/n
    p_tilde = (phat + z**2/(2*n)) / denom
    ME = (z/denom) * np.sqrt(phat*(1-phat)/n + z**2/(4*n**2))
    ci99 = (max(0, p_tilde - ME), min(1, p_tilde + ME))
    print(f"Tỷ lệ (smokers)={phat:.3f} với n={n},99% CI = [{ci99[0]:.3f}, {ci99[1]:.3f}]")
else:
    print("Không có quan sát trong nhóm hút thuốc")

NameError: name 'stats' is not defined

In [7]:
#cau f
inc_mask = (df["DAY 2 - 12AM"] > df["DAY 1 - 8AM"])
n = len(df)
k = inc_mask.sum()
phat = k/n
p0 = 0.65
alpha = 0.05
z = (phat - p0)/np.sqrt(p0*(1-p0)/n)

pval_two = 2*(1 - stats.norm.cdf(abs(z)))
print(f"Tỷ lệ tăng = {phat:.3f} (k={k}/{n}), z = {z:.3f}, p-value (hai phía) = {pval_two:.4f}")
if pval_two < alpha:
    print("bác bỏ H0: p ≠ 0.65 có ý nghĩa thống kê ")
else:
    print("khong đủ bằng chứng bác bỏ H0 ở mức 5%")


pval_one = 1 - stats.norm.cdf(z)
print(f"p-value ( H1: p > 0.65) = {pval_one:.4f}")

NameError: name 'stats' is not defined

In [8]:
#cau g
x = df["DAY 2 - 8AM"].to_numpy()
n = len(x)
mean = np.mean(x)
std = np.std(x, ddof=1)
mu0 = 98
tstat = (mean - mu0)/(std/np.sqrt(n))
alpha = 0.01
pval_one = 1 - stats.t.cdf(tstat, df=n-1)
print(f"Mean DAY 2 - 8AM = {mean:.3f}, t = {tstat:.3f}, p-value (một phía) = {pval_one:.4f}")
if pval_one < alpha:
    print("Bác bỏ h0: có bằng chứng ở mức 1% rằng nhiệt độ trung bình > 98")
else:
    print("Không đủ bằng chứng ở mức 1% để khẳng định nhiệt độ trung bình > 98")

NameError: name 'stats' is not defined